# 03 — Machine Learning
Test data is not used for model, hyperparameter, or threshold selection.

In [3]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV,cross_val_score,cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score
df=pd.read_csv("appointments_clean.csv")

## Task 3.1 — Features and split

In [4]:
X=df[["age","distance_km","fee_inr","lead_days","prior_no_shows","reminder_sent"]].copy()
X["reminder_sent"]=X["reminder_sent"].map({"Yes":1,"No":0})
X=pd.concat([X,pd.get_dummies(df[["booking_channel","visit_type"]],columns=["booking_channel","visit_type"],dtype=int)],axis=1)
y=df["no_show"]
assert X.shape[1]==11
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
print("Features:",X.columns.tolist(),"Train:",X_train.shape,"Test:",X_test.shape)
print("Test no-shows:",int(y_test.sum()))
print("Always-zero baseline accuracy:",accuracy_score(y_test,np.zeros(len(y_test))))

Features: ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up'] Train: (792, 11) Test: (198, 11)
Test no-shows: 49
Always-zero baseline accuracy: 0.7525252525252525


## Task 3.2 — Logistic Regression

In [5]:
scaler=StandardScaler()
Xtr=scaler.fit_transform(X_train); Xte=scaler.transform(X_test)
lr=LogisticRegression(max_iter=1000).fit(Xtr,y_train)
p=lr.predict_proba(Xte)[:,1]; pred=(p>=.5).astype(int)
print("Accuracy:",accuracy_score(y_test,pred))
print("Confusion matrix:\\n",confusion_matrix(y_test,pred))
print("Precision:",precision_score(y_test,pred,zero_division=0))
print("Recall:",recall_score(y_test,pred,zero_division=0))
print("F1:",f1_score(y_test,pred,zero_division=0))
print("ROC-AUC:",roc_auc_score(y_test,p))
print(pd.DataFrame({"feature":X.columns,"coefficient":lr.coef_[0]}).to_string(index=False))
i=list(X.columns).index("reminder_sent")
print("exp(coef):",np.exp(lr.coef_[0][i]))
or_actual=np.exp(lr.coef_[0][i]/scaler.scale_[i])
print("exp(coef / scale):",or_actual)
print(f"In this model, holding other features constant, Yes reminder is observationally associated with {(1-or_actual)*100:.1f}% lower no-show odds than No; this is not causal.")

Accuracy: 0.7878787878787878
Confusion matrix:\n [[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1: 0.4473684210526316
ROC-AUC: 0.788933022873579
                feature  coefficient
                    age    -0.253880
            distance_km     0.253448
                fee_inr     0.452761
              lead_days     0.614471
         prior_no_shows     0.649612
          reminder_sent    -0.580669
    booking_channel_App    -0.046804
  booking_channel_Phone     0.215881
booking_channel_Walk-In    -0.186364
visit_type_Consultation     0.015658
   visit_type_Follow-up    -0.015658
exp(coef): 0.559524141750829
exp(coef / scale): 0.3093980903124433
In this model, holding other features constant, Yes reminder is observationally associated with 69.1% lower no-show odds than No; this is not causal.


## Task 3.3 — Trees and overfitting

In [6]:
for name,tree in [("Unrestricted",DecisionTreeClassifier(random_state=42)),
                   ("Depth 3",DecisionTreeClassifier(max_depth=3,random_state=42))]:
 tree.fit(X_train,y_train)
 print(name,"train accuracy",accuracy_score(y_train,tree.predict(X_train)),
       "test accuracy",accuracy_score(y_test,tree.predict(X_test)))
print("A large train–test gap in the unrestricted tree indicates overfitting.")
tree3=DecisionTreeClassifier(max_depth=3,random_state=42).fit(X_train,y_train)

Unrestricted train accuracy 1.0 test accuracy 0.6717171717171717
Depth 3 train accuracy 0.821969696969697 test accuracy 0.7777777777777778
A large train–test gap in the unrestricted tree indicates overfitting.


## Task 3.4 — Random Forest and comparison

In [7]:
rf=RandomForestClassifier(n_estimators=100,random_state=42).fit(X_train,y_train)
rfp=rf.predict_proba(X_test)[:,1]; rfpred=(rfp>=.5).astype(int)
print("RF accuracy",accuracy_score(y_test,rfpred),"recall",recall_score(y_test,rfpred),
      "ROC-AUC",roc_auc_score(y_test,rfp))
print(pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False).head(3))
def metrics_from_prob(prob,truth):
 pred=(prob>=.5).astype(int)
 return [accuracy_score(truth,pred),precision_score(truth,pred,zero_division=0),
         recall_score(truth,pred,zero_division=0),f1_score(truth,pred,zero_division=0),
         roc_auc_score(truth,prob)]
comparison=pd.DataFrame([
 ["Logistic Regression",*metrics_from_prob(p,y_test)],
 ["Depth-3 Tree",*metrics_from_prob(tree3.predict_proba(X_test)[:,1],y_test)],
 ["Random Forest",*metrics_from_prob(rfp,y_test)]
],columns=["model","accuracy","precision","recall","f1","roc_auc"])
print(comparison.to_string(index=False))

RF accuracy 0.7575757575757576 recall 0.3469387755102041 ROC-AUC 0.7057252431173813
lead_days      0.219905
age            0.196788
distance_km    0.194739
dtype: float64
              model  accuracy  precision   recall       f1  roc_auc
Logistic Regression  0.787879   0.629630 0.346939 0.447368 0.788933
       Depth-3 Tree  0.777778   0.647059 0.224490 0.333333 0.699699
      Random Forest  0.757576   0.515152 0.346939 0.414634 0.705725


## Task 3.5 — CV tuning, threshold selection, final test

In [8]:
grid=GridSearchCV(RandomForestClassifier(random_state=42),
 {"n_estimators":[100,200],"max_depth":[3,5,8]},cv=5,scoring="roc_auc",n_jobs=-1)
grid.fit(X_train,y_train)
print("Best params:",grid.best_params_,"Best CV AUC:",grid.best_score_)
lr_pipe=Pipeline([("scaler",StandardScaler()),("model",LogisticRegression(max_iter=1000))])
candidates={"lr_pipe":lr_pipe,"depth3":DecisionTreeClassifier(max_depth=3,random_state=42),
 "default_rf":RandomForestClassifier(n_estimators=100,random_state=42),"tuned_rf":grid.best_estimator_}
cv_scores={}
for name,est in candidates.items():
 scores=cross_val_score(est,X_train,y_train,cv=5,scoring="roc_auc")
 cv_scores[name]=scores
 print(name,"CV AUC mean",scores.mean(),"std",scores.std())
print("CV winner:",max(cv_scores,key=lambda k:cv_scores[k].mean()))
oof=cross_val_predict(lr_pipe,X_train,y_train,cv=5,method="predict_proba")[:,1]
for threshold in [.5,.35]:
 flag=(oof>=threshold).astype(int)
 print("threshold",threshold,"precision",precision_score(y_train,flag,zero_division=0),
       "recall",recall_score(y_train,flag),"F1",f1_score(y_train,flag),"flagged",flag.sum())
print("Recall is prioritized to identify more potentially missed appointments, subject to staff call capacity.")
lr_pipe.fit(X_train,y_train)
test_prob=lr_pipe.predict_proba(X_test)[:,1]
test_flag=(test_prob>=.35).astype(int)
print("Final test precision",precision_score(y_test,test_flag,zero_division=0))
print("Final test recall",recall_score(y_test,test_flag))
print("Final test flagged",test_flag.sum(),"ROC-AUC",roc_auc_score(y_test,test_prob))
pd.DataFrame({"appointment_id":df.loc[X_test.index,"appointment_id"].values,
 "no_show":y_test.values,"risk_prob":test_prob,"flag":test_flag}).to_csv("test_risk_scores.csv",index=False)
print("Clinics should calibrate thresholds against front-desk call capacity and the cost of empty slots.")

Best params: {'max_depth': 3, 'n_estimators': 200} Best CV AUC: 0.7673242251739405
lr_pipe CV AUC mean 0.7737139899948249 std 0.013673720786272386
depth3 CV AUC mean 0.6784494968661952 std 0.03954229944534606
default_rf CV AUC mean 0.7466024782933701 std 0.029477375006868034
tuned_rf CV AUC mean 0.7673242251739405 std 0.024484419922501832
CV winner: lr_pipe
threshold 0.5 precision 0.5753424657534246 recall 0.2485207100591716 F1 0.34710743801652894 flagged 73
threshold 0.35 precision 0.512987012987013 recall 0.46745562130177515 F1 0.4891640866873065 flagged 154
Recall is prioritized to identify more potentially missed appointments, subject to staff call capacity.
Final test precision 0.5714285714285714
Final test recall 0.5714285714285714
Final test flagged 49 ROC-AUC 0.788933022873579
Clinics should calibrate thresholds against front-desk call capacity and the cost of empty slots.
